# 03 · Dense NN, vanilla RNN y LSTM — Sentiment analysis (GoEmotions)
**Responsable:** Camila · Usa el split ya generado (`data/processed/train.csv` y `test.csv`, 800/200).
El test **solo** se usa para la evaluación final; el tuning se hace con validación cruzada dentro de train.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from src.train import (load_tokens, grid_search, final_runs, MODEL_NAMES, GRID, FIXED, log, SEED)
from src.evaluate import METRICS_DIR

FIG_DIR = ROOT / "outputs" / "figures"; FIG_DIR.mkdir(parents=True, exist_ok=True)
QUICK = False   # True = grid reducido para probar que todo corre (unos minutos menos)

## 1. Datos

In [ ]:
tok_tr, y_tr, tok_te, y_te = load_tokens("clean")
lens = [len(t) for t in tok_tr]
print(f"train={len(tok_tr)} test={len(tok_te)} | balance train={np.bincount(y_tr)} test={np.bincount(y_te)}")
print(f"tokens por comentario (clean): media={np.mean(lens):.1f} máx={max(lens)} vacíos={sum(l==0 for l in lens)}")

## 2. Tuning (CV estratificada de 3 folds sobre train)
Se exploran `hidden`, `batch_size` y `lr`; el número de `epochs` lo decide el early stopping (se reporta la época media óptima).
`emb_dim=64` y `dropout=0.5` quedan fijos para mantener el grid manejable.

In [ ]:
grid = {"hidden":[32,128], "batch_size":[32], "lr":[1e-3]} if QUICK else GRID
search = {}
for name in MODEL_NAMES:
    search[name] = grid_search(name, tok_tr, y_tr, grid=grid)
    search[name].to_csv(ROOT/"outputs"/"metrics"/f"gridsearch_{name}.csv", index=False)
    print("\n==", name); display(search[name].head(5).round(4))

## 3. Evaluación final en test (5 semillas, mejores hiperparámetros)

In [ ]:
final, runs = {}, {}
for name in MODEL_NAMES:
    best = search[name].iloc[0]
    params = {**FIXED, "hidden": int(best.hidden), "batch_size": int(best.batch_size), "lr": float(best.lr)}
    final[name], runs[name] = final_runs(name, tok_tr, y_tr, tok_te, y_te, params)
    print(name, params)

## 4. Tabla comparativa (incluye baseline)

In [ ]:
rows = []
for f in ["baseline_dummy"] + MODEL_NAMES:
    d = json.loads((METRICS_DIR/f"{f}.json").read_text())
    r = {"model": d["model"]}
    for k in ["accuracy","precision","recall","f1","kappa"]:
        s = d.get("std", {}).get(k)
        r[k] = f"{d[k]:.3f}" + (f" ± {s:.3f}" if s is not None else "")
    r["params"] = d.get("n_params", "-"); r["sec/run"] = d.get("train_seconds_per_run", "-")
    rows.append(r)
table = pd.DataFrame(rows); table.to_csv(METRICS_DIR/"comparison.csv", index=False)
table

## 5. Figuras

In [ ]:
# Curvas de entrenamiento (semilla 42)
fig, ax = plt.subplots(1, 3, figsize=(14, 3.6), sharey=True)
for a, name in zip(ax, MODEL_NAMES):
    h = pd.DataFrame(runs[name][0]["history"])
    a.plot(h.epoch, h.train_loss, label="train loss"); a.plot(h.epoch, h.val_loss, label="val loss")
    a.axvline(runs[name][0]["best_epoch"], ls="--", c="gray", lw=1)
    a.set_title(name); a.set_xlabel("epoch")
ax[0].set_ylabel("BCE loss"); ax[0].legend(); plt.tight_layout()
plt.savefig(FIG_DIR/"nn_training_curves.png", dpi=200); plt.show()

In [ ]:
# Matrices de confusión (promedio de semillas)
fig, ax = plt.subplots(1, 3, figsize=(12, 3.6))
for a, name in zip(ax, MODEL_NAMES):
    cm = np.array(final[name]["confusion_matrix"])
    a.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2): a.text(j, i, f"{cm[i,j]:.1f}", ha="center", va="center")
    a.set_xticks([0,1], ["neg","pos"]); a.set_yticks([0,1], ["neg","pos"])
    a.set_title(name); a.set_xlabel("predicho"); a.set_ylabel("real")
plt.tight_layout(); plt.savefig(FIG_DIR/"nn_confusion_matrices.png", dpi=200); plt.show()

In [ ]:
# Comparación de métricas con barras de error (baseline sin error)
mets = ["accuracy","precision","recall","f1","kappa"]
names = ["baseline_dummy"] + MODEL_NAMES
data = {n: json.loads((METRICS_DIR/f"{n}.json").read_text()) for n in names}
x = np.arange(len(mets)); w = 0.2
fig, ax = plt.subplots(figsize=(9, 4))
for i, n in enumerate(names):
    ax.bar(x + (i-1.5)*w, [data[n][m] for m in mets], w, label=n,
           yerr=[data[n].get("std", {}).get(m, 0) for m in mets], capsize=2)
ax.set_xticks(x, mets); ax.set_ylim(0, 1); ax.legend(); plt.tight_layout()
plt.savefig(FIG_DIR/"nn_metrics_comparison.png", dpi=200); plt.show()

## 6. (Opcional) Ablación: ¿ayuda conservar las stopwords?
El pipeline del equipo elimina stopwords, pero las RNN/LSTM aprovechan el orden y los conectores. Se reentrenan con `text_mode="full"` y los mismos hiperparámetros.

In [ ]:
ftr, fy_tr, fte, fy_te = load_tokens("full")
for name in ["vanilla_rnn", "lstm"]:
    p = final[name]["params"]
    final_runs(name, ftr, fy_tr, fte, fy_te, p, save_name=f"{name}_full")

## 7. Notas para el análisis (completar con los resultados reales)
- ¿Qué modelo gana en F1/kappa y por cuánto respecto al baseline (kappa ≈ 0)? ¿La diferencia supera la desviación entre semillas?
- Dense (sin orden) vs RNN/LSTM (con orden): ¿importa el orden en comentarios de ~5–13 palabras?
- RNN vanilla vs LSTM: gradientes y memoria a largo plazo; con secuencias tan cortas la ventaja puede ser pequeña.
- Sobreajuste: 800 ejemplos, vocabulario de miles de palabras → observar la brecha train/val en las curvas.
- Costo: nº de parámetros y segundos por corrida (ver tabla).